## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 🔌 Azure AI Agent with Hosted MCP Tools

This notebook demonstrates integration of Azure AI Agents with hosted Model Context Protocol (MCP) servers using `FoundryAgent`, including user approval workflows for function call security.

## Features Covered:
- Setting up Azure AI Agents with Hosted MCP tools (configured server-side)
- Connecting to external MCP servers for enhanced capabilities
- Implementing user approval workflows for secure function calls
- Thread-based conversation management
- Querying Microsoft Learn documentation via MCP

### ⚠️ Important Note ⚠️
> **MCP (Model Context Protocol) allows agents to access external tools and services. User approval workflows ensure secure function execution.**

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure AI Project**: Access to an Microsoft Foundry project with deployed models
2. **Authentication**: Azure CLI installed and authenticated (`az login --use-device-code`)
3. **Environment Variables**: Set up your `.env` file with:
   - `AI_FOUNDRY_PROJECT_ENDPOINT`
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME`
4. **Dependencies**: Required agent-framework packages installed

If you need to use a different tenant:
```bash
az login --tenant <tenant-id>
```

## Import Libraries

Import the required libraries using the `FoundryAgent` API:

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import asyncio
import os
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Any

from agent_framework import Agent, Message
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.in").is_file())
load_dotenv(repo_root / ".env", override=False)

endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT") or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT") or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
if not endpoint or not model:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), "Select the repository .venv kernel."

# Public Microsoft Learn MCP server used by both examples below.
MCP_SERVER_NAME = "microsoft_learn"
MCP_SERVER_URL = "https://learn.microsoft.com/api/mcp"

print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print("Project endpoint and model: configured (values hidden)")

## Define an Auto-Approval Handler 🔐

When an MCP tool is configured with `approval_mode="always_require"`, the service pauses and returns
**approval requests** (`response.user_input_requests`) instead of executing the tool. The application must
respond with an approval decision before the run continues.

Interactive `input()` prompts do not work reliably inside Jupyter, so the helper below **auto-approves**
every pending request so the notebook runs end-to-end. In production you would surface each request to a
human and call `request.to_function_approval_response(approved)` with their decision.

In [ ]:
async def run_with_auto_approval(agent: Agent, query: str) -> Any:
    """Run the agent and automatically approve any MCP tool-call approval requests.

    A session keeps the conversation state, so each follow-up only needs to send the
    approval responses. Every request is approved here for demonstration; a production
    app should ask a human before approving.
    """
    session = agent.create_session()
    async with asyncio.timeout(180):
        result = await agent.run(query, session=session)
        while result.user_input_requests:
            approvals: list[Message] = []
            for request in result.user_input_requests:
                print(
                    f"🔐 Approval requested for '{request.function_call.name}' "
                    f"with arguments: {request.function_call.arguments}"
                )
                approvals.append(
                    Message(role="user", contents=[request.to_function_approval_response(True)])
                )
            result = await agent.run(approvals, session=session)
    return result

## Create an Agent with a Hosted MCP Tool 🔌

An MCP (Model Context Protocol) tool lets the agent call an external MCP server for extra capabilities.
Here the agent connects to the public **Microsoft Learn MCP** endpoint to answer documentation questions.

**Key Concepts (Agent Framework 1.17.0):**
- Build the hosted MCP tool with `FoundryChatClient.get_mcp_tool(name=..., url=...)`.
- Attach it to an application-owned `Agent(client=FoundryChatClient(...), tools=[mcp_tool])`.
- `approval_mode="never_require"` executes tool calls automatically; `"always_require"` returns approval
  requests you must respond to (see the auto-approval helper above).
- No private client subclass or tool-schema workaround is needed — public APIs accept the tool directly.

In [ ]:
async def run_hosted_mcp_without_approval() -> None:
    """Hosted MCP tool with automatic execution (no approval required)."""
    print("=== 🔌 MCP without approvals ===")

    mcp_tool = FoundryChatClient.get_mcp_tool(
        name=MCP_SERVER_NAME,
        url=MCP_SERVER_URL,
        approval_mode="never_require",
    )

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="LearnDocsAgent",
                instructions="You are a helpful assistant that can help with Microsoft documentation questions.",
                tools=[mcp_tool],
            )
            print(f"✅ Created agent: {agent.name}")
            print("🔌 MCP Tool: Microsoft Learn MCP (never_require approval)\n")

            query = "How to create an Azure storage account using az cli?"
            print(f"🤔 User: {query}")
            async with asyncio.timeout(120):
                result = await agent.run(query)
            assert result.text, "The service returned no answer."
            print(f"📚 {agent.name}: {result.text}\n")
        finally:
            await client.client.close()
            await client.project_client.close()

## Execute Without Approval 🚀

Run the example without approval mode - the agent will execute MCP tool calls automatically:

In [ ]:
# Run without approval mode
await run_hosted_mcp_without_approval()

## MCP with Approval Mode 🔐

With `approval_mode="always_require"`, each MCP tool call must be approved before it runs. The example
below uses the auto-approval helper so it completes without interactive input; in production you would
present each request to a human and approve or reject it explicitly.

In [ ]:
async def run_hosted_mcp_with_approval() -> None:
    """Hosted MCP tool that requires approval for each call; auto-approved for this demo."""
    print("=== 🔐 MCP with approvals ===")

    mcp_tool = FoundryChatClient.get_mcp_tool(
        name=MCP_SERVER_NAME,
        url=MCP_SERVER_URL,
        approval_mode="always_require",
    )

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="LearnDocsApprovalAgent",
                instructions="You are a helpful assistant that can use MCP tools to assist users.",
                tools=[mcp_tool],
            )
            print(f"✅ Created agent: {agent.name}")
            print("🔌 MCP Tool: Microsoft Learn MCP (always_require approval)\n")

            query = "Search Microsoft Learn for how to create an Azure resource group with az cli and summarize the steps."
            print(f"🤔 User: {query}")
            result = await run_with_auto_approval(agent, query)
            assert result.text, "The service returned no answer."
            print(f"📚 {agent.name}: {result.text}\n")
        finally:
            await client.client.close()
            await client.project_client.close()


await run_hosted_mcp_with_approval()

## Key Takeaways 📚

### Hosted MCP Tools

```python
from agent_framework.foundry import FoundryAgent

# MCP tools are configured server-side on the Foundry agent
agent = FoundryAgent(
    agent_name="my-mcp-agent",
    project_endpoint=endpoint,
    credential=AzureCliCredential(),
    instructions="...",
    client_type=FoundryAgentChatClient,
)
```

### User Approval Workflow

1. **Request Detection**: Check `result.user_input_requests` for pending approvals
2. **User Prompt**: Display function name and arguments to user
3. **Approval Response**: Use `user_input_needed.create_response(True/False)`
4. **Re-run Agent**: Continue execution with approval responses

### Benefits of MCP Integration

| Feature | Benefit |
|---------|----------|
| External Services | Access documentation, APIs, and tools |
| Security | User approval workflow for sensitive operations |
| Flexibility | Connect to any MCP-compatible server |
| Context | Thread-based conversation management |

### Approval Modes

Approval modes are configured server-side on the Foundry agent:

| Mode | Description |
|------|-------------|
| Never require | Tool calls execute automatically without user approval |
| Always require | Every tool call requires explicit user approval |

### Available MCP Endpoints

- **Microsoft Learn**: `https://learn.microsoft.com/api/mcp` - Documentation search and retrieval
- **Azure REST API Specs**: `https://gitmcp.io/Azure/azure-rest-api-specs` - Azure API specifications
- Custom MCP servers can be hosted for your specific needs

### Best Practices

1. **Security First**: Always implement approval workflows for sensitive operations
2. **Thread Management**: Use threads for multi-turn conversations
3. **Error Handling**: Handle network failures and approval denials gracefully
4. **Logging**: Log all function calls and approvals for audit purposes
5. **Agent Names**: Use hyphens (not underscores) in agent names

⚠️ **Security Note**: Configure approval mode server-side. For production, prefer always-require approval.